In [ ]:

import threading
import queue
import time

# Настройки проекта
TOTAL_ORDERS = 1000
CASHIERS_COUNT = 3
PRINTERS_COUNT = 2
BUFFER_SIZE = 5

# Буфер максимум на 5 заказов
order_queue = queue.Queue(maxsize=BUFFER_SIZE)

# Общие данные
printed_orders = []
results_lock = threading.Lock()
print_lock = threading.Lock()

# Статистика
cashier_stats = {i: 0 for i in range(1, CASHIERS_COUNT + 1)}
printer_stats = {i: 0 for i in range(1, PRINTERS_COUNT + 1)}


def cashier(cashier_id, orders):
    """Касса создает заказы и кладет их в очередь."""

    for order_id in orders:
        order = {
            "order_id": order_id,
            "cashier_id": cashier_id
        }

        # Если очередь заполнена, касса подождет
        order_queue.put(order)

        cashier_stats[cashier_id] += 1

        with print_lock:
            if order_id <= 10:
                print(
                    f"Касса {cashier_id}: "
                    f"заказ №{order_id} добавлен в очередь"
                )

    with print_lock:
        print(
            f"Касса {cashier_id} завершила работу. "
            f"Заказов: {cashier_stats[cashier_id]}"
        )


def printer(printer_id):
    """Принтер берет заказы из очереди и печатает билеты."""

    while True:
        order = order_queue.get()

        try:
            # None означает команду остановки
            if order is None:
                return

            # Имитация печати билета
            time.sleep(0.001)

            with results_lock:
                printed_orders.append(order["order_id"])
                printer_stats[printer_id] += 1

            with print_lock:
                if order["order_id"] <= 10:
                    print(
                        f"Принтер {printer_id}: "
                        f"билет №{order['order_id']} напечатан"
                    )

        finally:
            # Сообщаем очереди, что заказ обработан
            order_queue.task_done()


def main():
    start_time = time.perf_counter()

    # Распределяем 1000 заказов между тремя кассами
    orders_per_cashier = [
        range(i, TOTAL_ORDERS + 1, CASHIERS_COUNT)
        for i in range(1, CASHIERS_COUNT + 1)
    ]

    # Запускаем два принтера
    printers = []

    for printer_id in range(1, PRINTERS_COUNT + 1):
        thread = threading.Thread(
            target=printer,
            args=(printer_id,)
        )
        printers.append(thread)
        thread.start()

    # Запускаем три кассы
    cashiers = []

    for cashier_id, orders in enumerate(
        orders_per_cashier, start=1
    ):
        thread = threading.Thread(
            target=cashier,
            args=(cashier_id, orders)
        )
        cashiers.append(thread)
        thread.start()

    # Ждем завершения всех касс
    for thread in cashiers:
        thread.join()

    # Отправляем принтерам команды остановки
    for _ in range(PRINTERS_COUNT):
        order_queue.put(None)

    # Ждем завершения принтеров
    for thread in printers:
        thread.join()

    elapsed_time = time.perf_counter() - start_time

    # Проверяем результат
    unique_orders = set(printed_orders)

    print("\n========== РЕЗУЛЬТАТ ==========")
    print(f"Всего заказов: {TOTAL_ORDERS}")
    print(f"Касс: {CASHIERS_COUNT}")
    print(f"Принтеров: {PRINTERS_COUNT}")
    print(f"Размер очереди: {BUFFER_SIZE}")
    print(f"Напечатано билетов: {len(printed_orders)}")
    print(f"Уникальных билетов: {len(unique_orders)}")
    print(f"Дубликатов: {len(printed_orders) - len(unique_orders)}")
    print(f"Время выполнения: {elapsed_time:.3f} сек.")

    print("\nЗаказы по кассам:")
    for cashier_id, count in cashier_stats.items():
        print(f"Касса {cashier_id}: {count}")

    print("\nБилеты по принтерам:")
    for printer_id, count in printer_stats.items():
        print(f"Принтер {printer_id}: {count}")

    # Итоговая проверка
    if (
        len(printed_orders) == TOTAL_ORDERS
        and len(unique_orders) == TOTAL_ORDERS
        and set(printed_orders) == set(range(1, TOTAL_ORDERS + 1))
    ):
        print("\nУСПЕХ: все 1000 заказов обработаны ровно один раз!")
    else:
        print("\nОШИБКА: обнаружены потерянные или повторные заказы.")


if __name__ == "__main__":
    main()
